# Session 12 — Capstone Part 2: Analyse and Conclude

**Goal of this session:** build connectivity networks for 50 people and test a real developmental hypothesis.

*Python for Neuroscience, session 12 of 12.*

## Why this matters

Last session we turned one brain into an array. Now we do it for fifty and ask a question that people actually publish on.

The question: does the default mode network become more internally coordinated between childhood and adulthood? Everything you need to answer it, you learned in the previous eleven sessions.

## The data

Extracting region time series for 50 participants takes a few minutes, so it has been done once and saved in this repo. Same dataset, same atlas, same masker as session 11.

The file holds a 3D array: participants, then timepoints, then regions.

In [ ]:
import os

REPO_RAW = "https://raw.githubusercontent.com/saeedrafsharx/python-for-neuroscience/main/data/"
DATA = "../data/" if os.path.exists("../data/participants.csv") else REPO_RAW
print("reading data from:", DATA)

In [ ]:
import io
import urllib.request

import numpy as np
import pandas as pd


def load_npz(name):
    """Load from the local data folder, or from GitHub if we are on Colab."""
    if DATA.startswith("http"):
        with urllib.request.urlopen(DATA + name) as response:
            return np.load(io.BytesIO(response.read()), allow_pickle=True)
    return np.load(DATA + name, allow_pickle=True)


d = load_npz("dev_fmri_timeseries.npz")

series = d["timeseries"]
group = d["group"]
age = d["age"]
labels = [str(x) for x in d["regions"]]
networks = [str(x) for x in d["networks"]]

print("time series:", series.shape, "= participants x timepoints x regions")
print("children:", (group == "child").sum(), " adults:", (group == "adult").sum())
print(f"age range: {age.min():.1f} to {age.max():.1f} years")

## One connectivity matrix per person

Session 10, applied 50 times. `np.corrcoef` wants regions on the rows, so we transpose each participant's array before correlating.

In [ ]:
conn = np.array([np.corrcoef(subject.T) for subject in series])
print("connectivity:", conn.shape, "= participants x regions x regions")

In [ ]:
import matplotlib.pyplot as plt

mean_child = conn[group == "child"].mean(axis=0)
mean_adult = conn[group == "adult"].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, matrix, title in zip(axes, [mean_child, mean_adult],
                             ["Children", "Adults"]):
    im = ax.imshow(matrix, cmap="RdBu_r", vmin=-0.8, vmax=0.8)
    ax.set_title(f"{title}, mean connectivity", fontsize=15)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.75, label="correlation")
plt.show()

Both matrices show block structure, which is the same modular organisation we built by hand in session 10, now emerging from real brains. The adult matrix looks a little more contrasted.

"Looks a little more" is not a finding. Let's measure it.

## The hypothesis

The default mode network is four regions in this atlas. We take the average correlation among those four regions within each participant, giving one number per person, and compare children against adults.

One number per participant, decided before looking. That is what keeps this a test rather than a fishing trip.

In [ ]:
dmn = [i for i, net in enumerate(networks) if net == "DMN"]
print("DMN regions:", [labels[i] for i in dmn])

rows, cols = np.triu_indices(len(dmn), k=1)      # upper triangle, no diagonal
dmn_blocks = conn[:, dmn][:, :, dmn]
dmn_strength = dmn_blocks[:, rows, cols].mean(axis=1)

print("one value per participant:", dmn_strength.shape)

In [ ]:
from scipy import stats

child = dmn_strength[group == "child"]
adult = dmn_strength[group == "adult"]

t_stat, p_value = stats.ttest_ind(adult, child)
pooled_sd = np.sqrt((child.var(ddof=1) + adult.var(ddof=1)) / 2)
cohens_d = (adult.mean() - child.mean()) / pooled_sd

print(f"children: n = {len(child)}, mean = {child.mean():.3f}, SD = {child.std(ddof=1):.3f}")
print(f"adults:   n = {len(adult)}, mean = {adult.mean():.3f}, SD = {adult.std(ddof=1):.3f}")
print(f"\nt = {t_stat:.2f}, p = {p_value:.5f}, Cohen's d = {cohens_d:.2f}")

Adults show stronger within-network default mode connectivity than children, and the effect is large. This matches a well-replicated developmental finding: the default mode network integrates over childhood and adolescence.

You just reproduced it, on open data, in about fifteen lines.

In [ ]:
import seaborn as sns

df = pd.DataFrame({"group": group, "dmn": dmn_strength, "age": age})

sns.set_theme(style="whitegrid", font_scale=1.2)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

sns.violinplot(data=df, x="group", y="dmn", hue="group", order=["child", "adult"],
               palette=["#a0aec0", "#2b6cb0"], legend=False, inner=None, ax=axes[0])
sns.stripplot(data=df, x="group", y="dmn", order=["child", "adult"],
              color="black", size=4, alpha=0.6, ax=axes[0])
axes[0].set_xlabel("")
axes[0].set_ylabel("within-DMN connectivity (mean r)")
axes[0].set_title(f"t = {t_stat:.2f}, p = {p_value:.4f}, d = {cohens_d:.2f}", fontsize=13)

sns.regplot(data=df, x="age", y="dmn", scatter_kws={"s": 30, "alpha": 0.7},
            line_kws={"color": "#2b6cb0"}, ax=axes[1])
axes[1].set_xlabel("age (years)")
axes[1].set_ylabel("")
axes[1].set_title("Same result, plotted against age", fontsize=13)

plt.tight_layout()
plt.show()

The right panel is worth a pause. Age is not two groups, it is a continuous variable, and the sample here is mostly young children with a cluster of adults at the far right. A straight line through that gap is doing a lot of guessing about the middle.

Splitting a continuous variable into groups always throws away information. Sometimes that is a fair trade. It is your job to say so out loud.

## The network view

Session 10 again, now on the group averages. Threshold, build the graph, measure the hubs.

In [ ]:
import networkx as nx

threshold = 0.45


def build_graph(matrix):
    adjacency = (matrix > threshold) & ~np.eye(matrix.shape[0], dtype=bool)
    graph = nx.from_numpy_array(adjacency.astype(int))
    return nx.relabel_nodes(graph, dict(enumerate(labels)))


g_child = build_graph(mean_child)
g_adult = build_graph(mean_adult)

for name, g in [("children", g_child), ("adults", g_adult)]:
    deg = dict(g.degree())
    top = sorted(deg.items(), key=lambda kv: -kv[1])[:5]
    print(f"{name:9s} edges = {g.number_of_edges():3d}  "
          f"mean degree = {np.mean(list(deg.values())):.1f}")
    print("          top hubs:", ", ".join(f"{r} ({d})" for r, d in top))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
pos = nx.spring_layout(build_graph((mean_child + mean_adult) / 2), seed=2)

for ax, g, title, colour in zip(axes, [g_child, g_adult],
                                ["Children", "Adults"], ["#a0aec0", "#2b6cb0"]):
    deg = dict(g.degree())
    nx.draw_networkx_edges(g, pos, alpha=0.35, ax=ax)
    nx.draw_networkx_nodes(g, pos, node_color=colour,
                           node_size=[deg[n] * 90 + 120 for n in g.nodes()], ax=ax)
    labels_to_show = {n: n for n in g.nodes() if deg[n] >= 4}
    nx.draw_networkx_labels(g, pos, labels=labels_to_show, font_size=9, ax=ax)
    ax.set_title(f"{title}, r > {threshold}, {g.number_of_edges()} edges", fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

Same threshold, same layout, same regions. The adult network carries more edges. That is the group difference we tested, drawn instead of tabulated.

## What you built over twelve sessions

You started with `firing_rate = 12.5`. You finished by downloading an open neuroimaging dataset, reducing it with a brain atlas, building a network per participant, and testing a developmental hypothesis with an effect size attached.

The path was: Python itself, then arrays and tables, then plots, then filtering, then statistics, then graphs, then real data. Nothing in the last two sessions used a tool you had not already met.

## Where to go next

**Packages worth your time**

- `MNE-Python` for EEG and MEG. Excellent tutorials, and the standard in the field.
- `Nilearn` for fMRI. You have already used it.
- `NiBabel` for reading neuroimaging file formats directly.
- `Brian2` or `NEST` if you want to simulate spiking neurons rather than analyse recordings.
- `SpikeInterface` for extracellular recordings and spike sorting.
- `NetworkX` and `graph-tool` for anything network related.

**Open data to practise on**

- OpenNeuro, for a very large collection of raw BIDS datasets.
- ABIDE and ADHD-200, reachable through `nilearn.datasets`.
- The Human Connectome Project, for high quality structural and functional data.
- CRCNS, for electrophysiology.

**Habits worth building**

Put your analysis in version control from day one. Set the random seed. Write down the threshold you chose and why. Plot the raw data before you plot the summary.

That last one has saved more analyses than any statistical test.

Thanks for following along.